# 올리브영 PDRN 제품 데이터 수집

본 노트북은 올리브영 플랫폼에서 PDRN 마케팅 제품의 전성분 및 리뷰 데이터를 수집하여 최종 데이터셋을 구성한 과정을 기록한다.

**수집 파이프라인 요약**
1. 상품 목록 수집
2. 상품 기본 정보 파싱·정제 및 CSV 저장
3. 리뷰 데이터 수집 (상품별 재구매 리뷰 비율 집계)
4. 전성분 데이터 수집
5. 최종 데이터셋 병합 및 저장

> **참고:** 이 레포에는 수집한 원본 데이터(CSV)와 수집 스크립트를 포함하지 않는다.
> 이 노트북은 수집 절차를 기록하고, 파싱·병합 등 전처리 코드만 담고 있다.

## 1. 상품 목록 수집

올리브영에서 'PDRN' 키워드로 검색한 결과 페이지 4개(총 159개 상품)를 대상으로 한다.

검색 결과 목록 페이지는 동적 렌더링 방식이므로, 브라우저에서 각 상품 카드의 URL과 화면에 표시된 텍스트를 직접 추출하여
`{"url": ..., "text": ...}` 형태의 리스트로 정리하였다.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")  # 원본 데이터는 레포에 포함되지 않음

# 검색 결과 4개 페이지에서 추출한 상품 카드 목록
# 형태: [{"url": "...goodsNo=A000000000000...", "text": "브랜드\n상품명\n...\n정가 판매가\n..."}, ...]
page_items = []  # 원본 데이터 미포함

## 2. 상품 목록 텍스트 파싱
 
 검색 결과 목록 페이지에서 각 상품의 텍스트를 직접 추출하여 파싱한다. 브랜드명, 상품명, 현재 판매가, 상품 ID를 추출하며, 불필요한 UI 텍스트(베스트, 세일, 증정 등)는 제거한다.
 

In [ ]:
BAD_WORDS = ["베스트", "세일", "증정", "오늘드림", "찜하기전", "무료배송"]

def parse_list_text(text):
    lines = [x.strip() for x in text.split("\n") if x.strip()]
    
    # 가격 줄 위치 찾기
    price_current = None
    price_idx = None
    
    for i, line in enumerate(lines):
        prices = re.findall(r"([\d,]+)원", line)
        if prices:
            price_current = int(prices[-1].replace(",", ""))
            price_idx = i
            break
    
    # 가격 전까지만 후보로 사용
    candidates = lines[:price_idx] if price_idx is not None else lines
    
    # 불필요 태그 제거
    candidates = [
        x for x in candidates
        if x not in BAD_WORDS
    ]
    
    brand_name = candidates[0] if len(candidates) >= 1 else None
    product_name = candidates[1] if len(candidates) >= 2 else None
    
    return {
        "brand_name": brand_name,
        "product_name": product_name,
        "price_current": price_current
    }

## 3. 페이지별 데이터 통합 및 정제

4개 페이지의 상품 목록을 하나로 통합하고, URL 기준으로 중복 상품을 제거한다. 또한 URL에서 정규식으로 product_id를 추출한다.

In [ ]:
df_pages = pd.DataFrame(page_items, columns=["url", "text"])

parsed = df_pages["text"].apply(parse_list_text).apply(pd.Series)
df_list = pd.concat([df_pages[["url"]], parsed], axis=1)

df_list = df_list.drop_duplicates("url").reset_index(drop=True)
df_list["product_id"] = df_list["url"].str.extract(r"goodsNo=([^&]+)")

print(len(df_list))  # 원본 실행 결과: 159

## 4. 상품 기본 정보 CSV 저장

정제된 상품 목록을 `oliveyoung_pdrn_list_basic.csv`로 저장한다. 이후 리뷰 수집 및 전성분 수집의 기준 파일로 활용된다.

In [ ]:
df_list.to_csv(DATA_DIR / "oliveyoung_pdrn_list_basic.csv", index=False, encoding="utf-8-sig")

## 5. 리뷰 데이터 수집

상품별로 리뷰를 조회하여 재구매 리뷰 비율을 집계하였다. 수집 코드는 레포에 포함하지 않는다.

**수집 방법**
- 상품별로 '도움돼요 순' 정렬 기준 상위 리뷰를 최대 500개(페이지당 50개 × 10페이지)까지 조회
- 각 리뷰에 표시된 재구매 여부를 집계
  - `total_reviews`: 조회된 리뷰 수 (최대 500)
  - `repurchase_reviews`: 그중 재구매로 표시된 리뷰 수
  - `repurchase_rate` = repurchase_reviews / total_reviews × 100
- 리뷰 본문, 작성자 정보 등은 저장하지 않고 **상품별 집계값만** 사용
- 서버 부담을 줄이기 위해 상품 간 요청에 2초 간격을 두었다

**수집 한계**
- 전체 리뷰가 500개를 초과하는 상품은 샘플 기반 추정값이며, 실제 재구매율과 차이가 있을 수 있음
- 정렬 기준이 '도움돼요 순'으로 고정되어 있어 샘플 편향이 존재할 수 있음. 이는 본 연구의 제한점으로 명시함

## 6. 전성분 데이터 수집

상품 상세 페이지의 상품정보 제공고시 중 '화장품법에 따라 기재해야 하는 모든 성분' 항목 텍스트를 상품 ID 기준으로 수집하였다.
수집은 브라우저에서 상품별로 순차 진행하였으며, 요청 간 0.5초 간격을 두었다. 수집 코드는 레포에 포함하지 않는다.

**수집 결과 및 한계**
- 전체 159개 상품 중 154개 전성분 수집 성공 (97%)
- 5개 상품은 상세 정보를 조회할 수 없어 수집 불가
- 수집된 전성분은 `ingredients.csv`, `ingredients_109.csv` 두 파일로 나뉘어 저장됨 (1차 수집 후 누락분 재수집)

## 7. 최종 데이터셋 구성 완료
 
상품 기본 정보, 리뷰 데이터, 전성분 데이터를 `product_id` 기준으로 병합하여 최종 데이터셋 `dataset.csv`를 생성하였다.
 
| 컬럼명 | 설명 |
|--------|------|
| `url` | 올리브영 상품 상세 페이지 URL |
| `brand_name` | 브랜드명 |
| `product_name` | 상품명 |
| `price_current` | 현재 판매가 (원) |
| `product_id` | 올리브영 상품 고유 ID |
| `total_reviews` | 샘플링된 전체 리뷰 수 (최대 500) |
| `repurchase_reviews` | 샘플 내 재구매 리뷰 수 |
| `repurchase_rate` | 재구매율 (%) |
| `ingredients_text` | 전성분 텍스트 |
 
**최종 데이터셋 현황**
- 총 상품 수: 159개
- 전성분 수집 완료: 154개 (97%)
- 결측값: ingredients_text 5개
 

In [ ]:
df_main = pd.read_csv(DATA_DIR / 'oliveyoung_pdrn_final.csv')
df_ing1 = pd.read_csv(DATA_DIR / 'ingredients.csv')
df_ing2 = pd.read_csv(DATA_DIR / 'ingredients_109.csv')


cols_to_drop = [c for c in ['ingredients_text', 'ingredients_x', 'ingredients_y', 'ingredient'] if c in df_main.columns]
if cols_to_drop:
    df_main = df_main.drop(columns=cols_to_drop)

df_ing_combined = pd.concat([df_ing1, df_ing2], ignore_index=True)
df_ing_combined = df_ing_combined.drop_duplicates(subset=['product_id'], keep='last')

df_merged = pd.merge(df_main, df_ing_combined, on='product_id', how='left')

if 'ingredient' in df_merged.columns:
    df_merged = df_merged.rename(columns={'ingredient': 'ingredients_text'})

requested_columns = [
    'url', 'brand_name', 'product_name', 'price_current', 'product_id', 
    'total_reviews', 'repurchase_reviews', 'repurchase_rate', 'ingredients_text'
]

final_columns = [col for col in requested_columns if col in df_merged.columns]
df_final = df_merged[final_columns]

df_final.to_csv(DATA_DIR / 'dataset.csv', index=False, encoding='utf-8-sig')

display(df_final.head())